In [1]:
import kagglehub
from numpy.ma.core import indices
from statsmodels.graphics.functional import rainbowplot

# Download latest version
path = kagglehub.dataset_download("harshshinde8/movies-csv")

print("Path to dataset files:", path)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/mihirmithani/.cache/kagglehub/datasets/harshshinde8/movies-csv/versions/1


In [2]:
import pandas as pd
import numpy as np
data=pd.read_csv('movies.csv')

In [3]:
data.describe()

,index,budget,id,popularity,revenue,runtime,vote_average,vote_count
count,4803.000000,4.803000e+03,4803.000000,4803.000000,4.803000e+03,4801.000000,4803.000000,4803.000000
mean,2401.000000,2.904504e+07,57165.484281,21.492301,8.226064e+07,106.875859,6.092172,690.217989
std,1386.651002,4.072239e+07,88694.614033,31.816650,1.628571e+08,22.611935,1.194612,1234.585891
min,0.000000,0.000000e+00,5.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000
25%,1200.500000,7.900000e+05,9014.500000,4.668070,0.000000e+00,94.000000,5.600000,54.000000
50%,2401.000000,1.500000e+07,14629.000000,12.921594,1.917000e+07,103.000000,6.200000,235.000000
75%,3601.500000,4.000000e+07,58610.500000,28.313505,9.291719e+07,118.000000,6.800000,737.000000
max,4802.000000,3.800000e+08,459488.000000,875.581305,2.787965e+09,338.000000,10.000000,13752.000000


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
data["genres"]=data["genres"].fillna("")
data["overview"]=data["overview"].fillna("")
data["content"]=data["genres"]+data["overview"]

In [6]:
tfidf=TfidfVectorizer(stop_words='english')
tfidf_matrix=tfidf.fit_transform(data["content"])

In [7]:
cosine_sim = cosine_similarity(tfidf_matrix,tfidf_matrix)

In [8]:
cosine_sim

array([[1.        , 0.03349705, 0.01667972, ..., 0.        , 0.        ,
        0.        ],
       [0.03349705, 1.        , 0.0105919 , ..., 0.02002761, 0.        ,
        0.        ],
       [0.01667972, 0.0105919 , 1.        , ..., 0.0142566 , 0.        ,
        0.        ],
       ...,
       [0.        , 0.02002761, 0.0142566 , ..., 1.        , 0.02339285,
        0.00671136],
       [0.        , 0.        , 0.        , ..., 0.02339285, 1.        ,
        0.01150822],
       [0.        , 0.        , 0.        , ..., 0.00671136, 0.01150822,
        1.        ]])

In [9]:
#dropping duplicate indices
indices=pd.Series(data.index,index=data["original_title"]).drop_duplicates()


In [10]:
def recommendation_movie(title):
    topn=5
    if title not in indices:
        return "Movie not found!!!!!!!!!!"
    idx=indices[title]
    sim_scores=list(enumerate(cosine_sim[idx]))
    sim_scores=sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores=sim_scores[1:topn+1]
    movie_indices=[i[0] for i in sim_scores]
    return data["original_title"].iloc[movie_indices]

In [11]:
for i in range(1,4,1):
    print(i,"  ")
    print(recommendation_movie("Avatar"),"\n")


1   
3604           Apollo 18
1482    The Darkest Hour
1341    Obitaemyy Ostrov
634           The Matrix
529     Tears of the Sun
Name: original_title, dtype: object 

2   
3604           Apollo 18
1482    The Darkest Hour
1341    Obitaemyy Ostrov
634           The Matrix
529     Tears of the Sun
Name: original_title, dtype: object 

3   
3604           Apollo 18
1482    The Darkest Hour
1341    Obitaemyy Ostrov
634           The Matrix
529     Tears of the Sun
Name: original_title, dtype: object 



In [12]:
##colabaretive filtering based recommendation system

In [13]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("luisreimberg/ratingscsv")

print("Path to dataset files:", path)

Path to dataset files: /Users/mihirmithani/.cache/kagglehub/datasets/luisreimberg/ratingscsv/versions/1


In [14]:
ratings=pd.read_csv('ratings.csv')

In [15]:
ratings.describe()

,userId,movieId,rating,timestamp
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,326.127564,19435.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,1.000000,1.000000,0.500000,8.281246e+08
25%,177.000000,1199.000000,3.000000,1.019124e+09
50%,325.000000,2991.000000,3.500000,1.186087e+09
75%,477.000000,8122.000000,4.000000,1.435994e+09
max,610.000000,193609.000000,5.000000,1.537799e+09


In [16]:
user_item=ratings.pivot_table(index='userId', columns='movieId', values='rating').fillna(0)

In [17]:
#similarity in between the users

In [26]:
user_similarity=cosine_similarity(user_item)
user_similarity_data=pd.DataFrame(user_similarity,index=user_item.index,columns=user_item.index)
user_similarity_data.describe()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
count,610.000000,610.000000,610.000000,610.000000,610.000000,610.000000,610.000000,610.000000,610.000000,610.000000,...,610.000000,610.000000,610.000000,610.000000,610.000000,610.000000,610.000000,610.000000,610.000000,610.000000
mean,0.134684,0.052725,0.010084,0.092815,0.124011,0.123589,0.136654,0.154505,0.044090,0.066918,...,0.136189,0.160800,0.113055,0.102135,0.100058,0.129797,0.133305,0.179964,0.113157,0.136639
std,0.083870,0.070423,0.042391,0.070298,0.119650,0.126842,0.083581,0.155218,0.055585,0.068791,...,0.105465,0.149669,0.081234,0.111721,0.067885,0.079248,0.082682,0.103584,0.120320,0.095415
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.012993,0.000000,0.000000,0.000000,0.000000
25%,0.080573,0.000000,0.000000,0.048826,0.044071,0.039298,0.077950,0.050781,0.000000,0.021018,...,0.048001,0.057638,0.061984,0.030277,0.055767,0.074589,0.069416,0.111745,0.035990,0.066873
50%,0.122640,0.029018,0.003531,0.077624,0.091391,0.076712,0.127143,0.108616,0.036904,0.043810,...,0.115704,0.114817,0.093717,0.064848,0.094301,0.111794,0.130759,0.158537,0.079649,0.118093
75%,0.172671,0.077870,0.008579,0.126982,0.151973,0.159861,0.185984,0.188414,0.072279,0.103943,...,0.213326,0.194683,0.144075,0.123708,0.133247,0.174050,0.182838,0.226191,0.129316,0.183908
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ratings["movieId"] = ratings["movieId"].fillna("")
ratings["rating"] = ratings["rating"].fillna("")
ratings["content"] = ratings["movieId"] + ratings["rating"]
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(data["content"])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
#dropping duplicate indices
indices = pd.Series(ratings.index, index=ratings["userId"]).drop_duplicates()


def recommendation_movie(title):
    topn = 5
    if title not in indices:
        return "Movie not found!!!!!!!!!!"
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:topn + 1]
    movie_indices = [i[0] for i in sim_scores]
    return ratings["userId"].iloc[movie_indices]


for i in range(1, 4, 1):
    print(i, "  ")
    print(recommendation_movie(2), "\n")

1   


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()